In [2]:
import os
from pathlib import Path

from adapters.cavia.cavia_scenario_loader import CaviaScenarioLoader
from adapters.cavia.find_valid_scenarios import find_or_load_scenarios, get_scenario_paths
from adapters.cavia.utils.path import PKL_PATH
from edge_sim_py.component_manager import ComponentManager
from edge_sim_py.components.data_packet import DataPacket
from edge_sim_py.simulator import Simulator


valid_scenarios = find_or_load_scenarios(PKL_PATH, force_rescan=False)

distributions = ["exponential", "gaussian", "uniform"]
BASE_SEED = 42
NUM_RUNS = 10

for dist_type in distributions:
    print(f"\nDISTRIBUTION: {dist_type}")
    for scenario_rel_path, apps in valid_scenarios.items():

        scenario_name = os.path.basename(scenario_rel_path)
        #print(f"\nSCENARIO: {scenario_name}")
        success_apps = []
        invalid_apps = []

        for app_name in apps:
            try:
                for run_index in range(NUM_RUNS):         
                    
                    phys_path, app_path, pkl_path = get_scenario_paths(scenario_name, app_name)
                    
                    current_seed = BASE_SEED + run_index

                    loader = CaviaScenarioLoader(
                        physical_graph_path=phys_path,
                        app_graph_path=app_path,
                        pkl_path=pkl_path,
                        seed=current_seed,
                        dist=dist_type,
                    ).build_scenario()
                    
                    dataset = ComponentManager.export_scenario(save_to_file=True, file_name=scenario_name)

                    # Cartella log specifica: logs/1_26_solution_v0/1SSS/
                    current_logs_dir = Path("logs") / dist_type / scenario_name / app_name / f"run_{run_index}"
                    current_logs_dir.mkdir(parents=True, exist_ok=True)
                    
                    def my_algorithm(parameters):
                        return
                    
                    def static_dummy_mobility(user):
                        user.coordinates_trace.append(user.coordinates)

                    simulator = Simulator(
                        dump_interval=1,
                        tick_unit="seconds",
                        tick_duration=1,
                        stopping_criterion = lambda model: all(d._status == 'finished' for d in DataPacket.all()) or model.schedule.steps >= 500,
                        resource_management_algorithm=my_algorithm,
                        user_defined_functions=[static_dummy_mobility],
                        logs_directory=str(current_logs_dir)
                    )
                
                
                    simulator.initialize(input_file=f"datasets/{scenario_name}.json")
                    simulator.run_model()
                
                #print(f"Completata: {app_name} (Log salvati in {current_logs_dir})")
                success_apps.append(app_name)

            except ValueError as e:
                #print(f"Skipping {app_name}: {e}")
                invalid_apps.append(app_name)
                continue

            except Exception as e:
                print(f"Errore durante la simulazione di {app_name}: {e}")
        
        total = len(apps)
        done = len(success_apps)
        print(f"SCENARIO: {scenario_name} | ({done}/{total}) Apps completate")
        if success_apps:
            print(f"  OK: {', '.join(success_apps)}")
        if invalid_apps:
            print(f"  Skip (Invalid): {', '.join(invalid_apps)}")
                

print("\n\n TUTTE LE SIMULAZIONI SONO STATE COMPLETATE.")



DISTRIBUTION: exponential
SCENARIO: 1_26_solution_v0 | (2/6) Apps completate
  OK: 1MMM, 1MMS
  Skip (Invalid): 1SMM, 1SMS, 1SSM, 1SSS
SCENARIO: 1_26_solution_v1 | (2/6) Apps completate
  OK: 1MMM, 1MMS
  Skip (Invalid): 1SMM, 1SMS, 1SSM, 1SSS
SCENARIO: 1_26_solution_v2 | (0/4) Apps completate
  Skip (Invalid): 1SMM, 1SMS, 1SSM, 1SSS
SCENARIO: 1_26_solution_v3 | (0/4) Apps completate
  Skip (Invalid): 1SMM, 1SMS, 1SSM, 1SSS
SCENARIO: 1_26_solution_v4 | (4/8) Apps completate
  OK: 1MMM, 1MMS, 1MSM, 1MSS
  Skip (Invalid): 1SMM, 1SMS, 1SSM, 1SSS
SCENARIO: 1_26_solution_v5 | (0/4) Apps completate
  Skip (Invalid): 1SMM, 1SMS, 1SSM, 1SSS
SCENARIO: 1_26_solution_v6 | (4/8) Apps completate
  OK: 1MMM, 1MMS, 1MSM, 1MSS
  Skip (Invalid): 1SMM, 1SMS, 1SSM, 1SSS
SCENARIO: 1_26_solution_v7 | (0/4) Apps completate
  Skip (Invalid): 1SMM, 1SMS, 1SSM, 1SSS
SCENARIO: 1_26_solution_v8 | (2/6) Apps completate
  OK: 1MMM, 1MMS
  Skip (Invalid): 1SMM, 1SMS, 1SSM, 1SSS
SCENARIO: 1_26_solution_v9 | (4/8) A